# Requirement 1: Administrator confidentiality

> Azure administrative privileges alone must not allow someone to obtain session files, code, results, or secrets, including while execution is running.

**Covered by:** Both

**How it is fulfilled:** Sandbox operations are data-plane operations that need the Entra role **Container Apps SandboxGroup Data Owner**. Only the orchestrator holds it. Administrators hold control-plane roles only, and a conditioned *Role Based Access Control Administrator* role stops them granting themselves Data Owner.

In [1]:
import sys
sys.path.insert(0, "..")

In [2]:
from helpers import *
import uuid
arm = token("admin", "arm")
data = token("admin", "data")
print("Identity: sp-contoso-admin (Contributor on the resource group + RBAC Administrator limited to Reader)")

Identity: sp-contoso-admin (Contributor on the resource group + RBAC Administrator limited to Reader)


In [3]:
# The admin can manage the group on the control plane
r = requests.get(f"{ARM}{GROUP_SCOPE}?api-version=2026-07-01", headers=arm, timeout=30)
check("control plane: admin reads the sandbox group (200)", r.status_code == 200)

PASS  control plane: admin reads the sandbox group (200)


In [4]:
# ...but cannot touch the data plane
r = requests.get(f"{DATA_GROUP_URL}/sandboxes?{DATA_API}", headers=data, timeout=30)
check(f"data plane: admin lists sandboxes -> {r.status_code} (403 expected)", r.status_code == 403)

PASS  data plane: admin lists sandboxes -> 403 (403 expected)


In [5]:
# The admin cannot grant itself the data-plane role
def grant(role_name):
    defs = requests.get(
        f"{ARM}/subscriptions/{SUBSCRIPTION}/providers/Microsoft.Authorization/roleDefinitions"
        f"?api-version=2022-04-01&$filter=roleName eq '{role_name}'", headers=arm, timeout=30).json()
    name = uuid.uuid5(uuid.NAMESPACE_DNS, "demo-" + role_name)  # fixed name keeps the demo repeatable
    body = {"properties": {"roleDefinitionId": defs["value"][0]["id"], "principalId": ADMIN_OID, "principalType": "ServicePrincipal"}}
    return requests.put(f"{ARM}{GROUP_SCOPE}/providers/Microsoft.Authorization/roleAssignments/{name}?api-version=2022-04-01",
                        headers=arm, json=body, timeout=30)

r = grant("Reader")
check(f"admin assigns itself Reader -> {r.status_code} (allowed by the condition; 409 means it already exists)", r.status_code in (200, 201, 409))
r = grant("Container Apps SandboxGroup Data Owner")
check(f"admin assigns itself Data Owner -> {r.status_code} (blocked by the condition)", r.status_code == 403)

PASS  admin assigns itself Reader -> 409 (allowed by the condition; 409 means it already exists)


PASS  admin assigns itself Data Owner -> 403 (blocked by the condition)


In [6]:
# The orchestrator, which holds Data Owner, can use the data plane
r = requests.get(f"{DATA_GROUP_URL}/sandboxes?{DATA_API}", headers=token("orchestrator", "data"), timeout=30)
check("orchestrator lists sandboxes (200)", r.status_code == 200)

PASS  orchestrator lists sandboxes (200)


**Caveat:** This holds only for identities that cannot assign roles on the group's scope. Anyone with Owner, User Access Administrator, or tenant-level elevation can grant themselves Data Owner. The scope must be owned by a separate security authority (see the Requirement 1 section of `sandbox-approach.md`).